# PFC production GLM

The mirror of `code/LEC_glm_production.ipynb`, and the other half of the cross-dataset
comparison.

**Two things are different here, both forced by the data, not chosen:**

1. **PFC can only fit the matched-13 design.** `build_data_dic_from_pfc` sets `HD_raw=None`
   (no head tracking) and there are no poke event tables, so `head_direction`,
   `poke_rewarded` and `poke_unrewarded` are unavailable — 16 − 3 = 13. This is also the
   *only* design that licenses a LEC-vs-PFC claim: CPD and Δr² are measured relative to the
   full model, so a design differing by 38 columns makes the two datasets' numbers
   non-comparable. Fit LEC with `--regset matched` for the comparison.
2. **There is no anatomy.** PFC has no `unit_regions`, so there is no regional split. The
   scientific payload of this notebook is section 3: the matched cross-dataset contrast.

Everything else — binned aggregation, LOSO CV, mixed reference coding, Δr² alongside CPD,
config-encoded section names — is deliberately identical, because the comparison only means
something if the two arms are the same analysis.

In [1]:
import os, sys, pickle, subprocess, time, shlex
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))

import glm_analysis_v2 as glm
import glm_cv as cv
import w1_refit as w
import run_glm_batch as rb

pd.set_option('display.width', 200)
glm.apply_gridmaze_style()

recdays = w.pfc_recdays()
print(f'{len(recdays)} PFC recdays; first 3: {recdays[:3]}')
print(f'matched design ({len(w.MATCHED_REGRESSORS)}): {w.MATCHED_REGRESSORS}')

25 PFC recdays; first 3: ['ab03_01092023_02092023', 'ab03_05092023_06092023', 'ab03_29082023_30082023']
matched design (13): ['place', 'task_state', 'goal_progress', 'goal_progress_distance', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'time_since_A', 'time_to_A', 'progress_since_A', 'distance_from_reward', 'distance_to_reward']


## 1. Configuration

Use the **same** width and scheme chosen in the LEC notebook. Deliberately not re-selected
here — picking configurations separately per dataset would make any difference between them
partly a difference of configuration.

In [2]:
RUN_MODE = 'load'                      # 'load' reads the cached fit; set 'slurm' to REFIT (submits 25 jobs)
WIDTH, SCHEME = 250, 'decile'           # <- must match the LEC run
SECTION, REGSET = 'all_regressors', 'matched'
PERMUTATIONS, CV_PERMS = 100, 100

sect = w.section_name(SECTION, width_ms=WIDTH, scheme=SCHEME, regset=REGSET)
cfg_flags = ['--section', SECTION, '--width-ms', str(WIDTH), '--scheme', SCHEME,
             '--regset', REGSET, '--permutations', str(PERMUTATIONS),
             '--cv-perms', str(CV_PERMS)]
print('section:', sect)

section: all_regressors__matched_250ms_decile


## 2. Production fit

PFC jobs are much lighter than LEC's: the data is a directory of per-recday `.npy` files, so
each job loads only its own recday instead of a 3.8 GB pickle.

In [3]:
REPO = os.path.abspath('../..')

if RUN_MODE == 'slurm':
    cmd = ['bash', 'sbatch_files/submit_glm_pfc.sh'] + cfg_flags
    print('$', ' '.join(shlex.quote(c) for c in cmd))
    print(subprocess.run(cmd, capture_output=True, text=True, cwd=REPO).stdout)
    while True:
        q = subprocess.run(['squeue', '--me', '--noheader'],
                           capture_output=True, text=True).stdout
        n = len([l for l in q.split('\n') if 'glm_' in l])
        print(f'  {n} job(s) still queued/running', flush=True)
        if n == 0:
            break
        time.sleep(60)
    subprocess.run([sys.executable, 'mFC_data/code/run_glm_batch.py', '--merge'] + cfg_flags,
                   cwd=REPO, check=True)

elif RUN_MODE == 'local':
    import argparse
    ap = argparse.ArgumentParser(); rb.add_config_args(ap)
    args = ap.parse_args(cfg_flags)
    for rd in recdays:
        rb.fit_one(rd, args)
    rb.merge(args)

elif RUN_MODE != 'load':
    raise ValueError(f'RUN_MODE must be slurm/local/load, got {RUN_MODE!r}')

$ bash sbatch_files/submit_glm_pfc.sh --section all_regressors --width-ms 250 --scheme decile --regset matched --permutations 100 --cv-perms 100
Submitting 25 job(s) with args: --section all_regressors --width-ms 250 --scheme decile --regset matched --permutations 100 --cv-perms 100
  submitted ab03_01092023_02092023 -> job 3510239
  submitted ab03_05092023_06092023 -> job 3510240
  submitted ab03_29082023_30082023 -> job 3510241
  submitted ah03_12082021_13082021 -> job 3510242
  submitted ah03_18082021_19082021 -> job 3510243
  submitted ah04_01122021_02122021 -> job 3510244
  submitted ah04_05122021_06122021 -> job 3510245
  submitted ah04_07122021_08122021 -> job 3510246
  submitted ah04_09122021_10122021 -> job 3510247
  submitted ah04_14122021_16122021 -> job 3510248
  submitted ah07_01092023_02092023 -> job 3510249
  submitted ah07_27082023_28082023 -> job 3510250
  submitted ah07_29082023_30082023 -> job 3510251
  submitted me08_06092021_09092021 -> job 3510252
  submitted me08

In [4]:
# load_glm_results returns a DICT keyed by artifact name (not a tuple).
res_pfc = glm.load_glm_results(rb.DEFAULT_OUT, sect, apply_exclusions=True, verbose=True)
cv_pfc  = res_pfc['cv_results']
print(f'{len(cv_pfc)} PFC recdays with cross-validated results')

25 PFC recdays with cross-validated results


## 3. LEC vs PFC, matched design

The point of this notebook. Both arms must be the **same section name** — same regressors,
same bin width, same placement scheme — or the contrast is confounded by configuration.

Recdays are not independent replicates in either dataset, so the unit of inference is the
**mouse** on both sides.

In [5]:
LEC_OUT = os.path.join(REPO, 'data', 'glm_outputs', 'LEC')
res_lec = glm.load_glm_results(LEC_OUT, sect, apply_exclusions=True, verbose=False)
cv_lec  = res_lec['cv_results']
print(f'LEC {len(cv_lec)} recdays   PFC {len(cv_pfc)} recdays   section {sect}')

# Refuse to compare arms that were not fitted identically.
assert set(cv_lec) and set(cv_pfc), 'both arms must be fitted before comparing'
regs_lec = set(next(iter(cv_lec.values()))['delta_r2_cv'])
regs_pfc = set(next(iter(cv_pfc.values()))['delta_r2_cv'])
assert regs_lec == regs_pfc, (
    f'designs differ — CPD is relative to the full model, so these are not comparable.\n'
    f'  LEC only: {sorted(regs_lec - regs_pfc)}\n  PFC only: {sorted(regs_pfc - regs_lec)}')
print(f'designs match on {len(regs_lec)} regressor groups')

load_glm_results('all_regressors__matched_250ms_decile'): no pickles found in /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/data/glm_outputs/LEC


KeyError: 'cv_results'

In [6]:
def per_mouse_delta_r2(cv_results, regressor):
    """Median delta_r2 per recday -> averaged within mouse. Mice are the unit of inference:
    a mouse's recdays share a probe and a brain, so pooling them is repeated measures."""
    per_rd = {rd: float(np.nanmedian(r['delta_r2_cv'][regressor]))
              for rd, r in cv_results.items() if regressor in r['delta_r2_cv']}
    df = pd.DataFrame({'recday': list(per_rd), 'value': list(per_rd.values())})
    df['mouse'] = df.recday.str.split('_').str[0]
    return df.groupby('mouse')['value'].mean()

rows = []
for g in sorted(regs_lec):
    if g.startswith('__'):
        continue
    a, b = per_mouse_delta_r2(cv_lec, g), per_mouse_delta_r2(cv_pfc, g)
    rows.append({'regressor': g,
                 'LEC': a.mean(), 'LEC_n_mice': len(a),
                 'PFC': b.mean(), 'PFC_n_mice': len(b),
                 'LEC - PFC': a.mean() - b.mean()})
comp = pd.DataFrame(rows).set_index('regressor').sort_values('LEC - PFC', ascending=False)
display(comp.round(5))

NameError: name 'regs_lec' is not defined

In [7]:
# Bootstrap the difference, resampling MICE on both sides
def boot_diff(g, n_boot=10000, seed=0):
    a, b = per_mouse_delta_r2(cv_lec, g).values, per_mouse_delta_r2(cv_pfc, g).values
    if len(a) < 2 or len(b) < 2:
        return None
    rng = np.random.default_rng(seed)
    d = np.array([rng.choice(a, len(a), True).mean() - rng.choice(b, len(b), True).mean()
                  for _ in range(n_boot)])
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {'diff': a.mean() - b.mean(), 'ci_lo': lo, 'ci_hi': hi,
            'p_boot': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())),
            'n_mice_LEC': len(a), 'n_mice_PFC': len(b)}

for g in ['place', 'task_state', 'goal_progress', 'time_from_reward']:
    r = boot_diff(g)
    if r:
        print(f"  {g:22s} LEC-PFC {r['diff']:+.5f}  CI[{r['ci_lo']:+.5f},{r['ci_hi']:+.5f}]"
              f"  n_mice {r['n_mice_LEC']}/{r['n_mice_PFC']}")

NameError: name 'cv_lec' is not defined

**Caveat to carry into any figure from this notebook.** The matched design excludes
`head_direction` and both poke regressors. In LEC, head direction was one of the regressors
that only became interpretable after the W0.2 fix and that binned aggregation rescued — so
the matched comparison necessarily leaves out one of the more interesting LEC results. Use
the full-16 LEC fit for within-LEC statements and this matched fit only for LEC-vs-PFC.

## 4. PFC on its own terms

Section 3 asks how PFC compares to LEC. This section asks what the PFC fit *is*, without
reference to the other dataset.

All panels use `delta_r2` rather than CPD (common denominator, so regressors are comparable
to each other) and average within mouse before pooling, showing the individual mice.

In [ ]:
import pfc_glm_plots as P
from importlib import reload; reload(P)

FIGDIR = '../data/figures/pfc_glm'
print(P.regressor_names(cv_pfc))

### 4.1 Does the model explain PFC at all?

The fraction of neurons with held-out R² above zero is the honest headline — a neuron below
zero is one the model fails to predict out of sample.

In [ ]:
P.plot_model_fit(cv_pfc, out_path=f'{FIGDIR}/pfc_model_fit.pdf');

### 4.2 What does PFC encode?

### How a $\Delta R^2$ bar is computed

Five steps, from the fit to the bar height. They are not interchangeable with the obvious
alternatives, so the exact chain matters.

**1. Per neuron, per fold** — `glm_cv.cv_scores`, leave-one-session-out. Sessions are
different *tasks*, so a fold tests generalisation across tasks. Each neuron's firing is
mean-centred **within session** first (`cv_center_within_sessions=True`), which grants a free
per-session offset so the CV tests tuning shape rather than absolute-rate stability. For each
held-out session, the full model and each reduced model (one regressor group dropped) are fit
on the *other* sessions and scored on the held-out one.

**2. Per neuron — accumulate, then divide once.** RSS and TSS are summed across folds and the
ratio is formed at the end:

$$\Delta R^2_g \;=\; \frac{\sum_{\text{folds}} \mathrm{RSS}_{\text{reduced},g} \;-\; \sum_{\text{folds}} \mathrm{RSS}_{\text{full}}}{\sum_{\text{folds}} \mathrm{TSS}}$$

This is **not** the mean of per-fold $\Delta R^2$. Pooling before the ratio is more stable,
and it matters here because sessions differ in length so folds differ in size.

Two consequences worth holding onto:

- $\Delta R^2$ can be **negative** — dropping a regressor can *improve* held-out prediction
  when it was only fitting noise. That is signal, and is not clipped.
- The denominator is **TSS, shared by every regressor**, so bars are comparable to each other
  and roughly additive toward the model's $R^2_{cv}$. CPD instead divides by each group's own
  reduced-model RSS, which inflates weak regressors when a dominant one stays in the model —
  by up to 12$\times$ in simulation, which is exactly this dataset's situation (place
  dominates).

**3. Per recday** — median across that recday's neurons.

**4. Per mouse** — mean across that mouse's recdays.

**5. Bar height** — mean across mice. The dots are the step-4 values, one per mouse.

Steps 3–4 are deliberately two steps (`per_mouse_stat`). Recdays of one mouse are the same
probe in the same brain, re-sorted, so they are repeated measures rather than replicates — and
they carry very different neuron counts (1 to 117). Pooling all of a mouse's neurons into one
median would let its biggest recday dominate. On this fit the two schemes differ by up to
33% (`acceleration`), 22% (`speed`), and `goal_progress` changes sign.

This is the same chain as `anatomy_split.per_mouse_effect`, which the regional panels use —
verified identical to 0.00e+00 — so the pooled and regional figures are computed the same way.


In [ ]:
P.plot_regressor_ranking(cv_pfc, out_path=f'{FIGDIR}/pfc_regressor_ranking.pdf');
display(P.summary_table(cv_pfc).round(5))

### 4.3 How many neurons, against their own null

The dashed line is chance. A bar sitting on it means that regressor is indistinguishable
from noise in PFC — which is a result, not a gap.

### The permutation null: why Freedman–Lane

**What we want to test.** For each neuron and each regressor *g*: *does g explain held-out
variance beyond what the other regressors already explain?* That "beyond the others" clause is
the whole content of a CPD or $\Delta R^2$ — they are **unique**-variance measures.

**Why shuffling the firing gets this wrong.** The obvious null is to circularly shift the
neuron's spike train and refit. But that destroys **every** regressor's relationship to firing,
not just *g*'s. So it tests the *global* hypothesis "nothing explains this neuron", which is a
different and much weaker question — and with `place` dominating, it is trivially rejectable
for reasons that have nothing to do with *g*.

The damage is concrete. Under the shuffle, no regressor explains anything, so the model's
**residual variance $\sigma^2$ is much larger** than in the real fit where place and speed have
absorbed variance. The full model always pays a parameter penalty of roughly $k\sigma^2$ for
its *k* extra columns — so under the shuffle that penalty is inflated, the null sits *further*
below zero than the observed value does, and the observed beats it almost every time.

Measured on synthetic data where *g* is **exactly null** while the other regressors carry real
signal (so the correct answer is 5%):

| null | frac p < 0.05, CPD | frac p < 0.05, $\Delta R^2$ |
|---|---|---|
| shuffle the firing | 0.133 | **1.000** |
| **Freedman–Lane** | 0.067 | **0.050** |
| permute *g*'s columns | 0.067 | 0.067 |

The shuffle null gives a **100% false-positive rate** on $\Delta R^2$. All three retain full
power (1.000) when *g* does carry signal, so this is a specificity failure, not a sensitivity
one.

**How Freedman–Lane works.** For each regressor *g*, per neuron:

1. Fit the **reduced** model — everything except *g* — to the real firing, giving a prediction
   $\hat{y}_{\text{red}}$ and residuals $e = y - \hat{y}_{\text{red}}$.
2. Circularly shift those residuals **within each session**, giving $e^*$.
3. Build surrogate data $y^* = \hat{y}_{\text{red}} + e^*$.
4. Run the **whole cross-validation** on $y^*$ — refit full and reduced on the training
   sessions, score on the held-out one — and compute the statistic exactly as for the real data.
5. Repeat 100 times; $p = (1 + \#\{\text{null} \ge \text{observed}\}) / (1 + n_{\text{perm}})$.

**Why that is the right null.** $y^*$ keeps the other regressors' structure intact (it is built
*from* their fit), so the surrogate data has the same residual variance as the real data and the
full model pays the **same** parameter penalty it pays on the real data. Only *g*'s relationship
to firing is destroyed. That is exactly the hypothesis a unique-variance measure is asking
about — and it is why the null lands at 0.050 instead of 1.000.

Two implementation details that matter:

- **Circular shifting, not free permutation.** Neighbouring time bins are strongly
  autocorrelated (the animal occupies a maze node for many bins). Freely permuting residuals
  would destroy that autocorrelation and make the null far too easy to beat. Shifting preserves
  it. Shifting happens **within session** for the same reason the design does — wrapping one
  task's residuals onto another task's regressors would be a different null again.
- **One shift set, drawn once**, reused across every fold and every model, so the null does not
  additionally average over shift noise.

**Cost.** Freedman–Lane needs a different $y^*$ per regressor, so it cannot share one permuted
trace across models the way the shuffle can — measured at ~246 min per LEC recday at
`n_perm=100`, against ~72 min for the shuffle. Run as one SLURM job per recday, so the wall
clock is a few hours rather than days.

In [ ]:
P.plot_significant_fraction(cv_pfc, out_path=f'{FIGDIR}/pfc_significant_fraction.pdf');

### 4.4 Mixed or specialised?

Per-neuron unique variance, and the count of regressors each neuron significantly encodes.
A population of specialists piles up at 1; mixed selectivity spreads right. The heatmap is
`RdBu_r` centered at zero because Δr² is signed — blue means dropping that regressor
*improved* held-out prediction for that neuron.

In [ ]:
P.plot_neuron_heatmap(cv_pfc, sort_by='place', out_path=f'{FIGDIR}/pfc_neuron_heatmap.pdf');
P.plot_mixed_selectivity(cv_pfc, out_path=f'{FIGDIR}/pfc_mixed_selectivity.pdf');

**What this fit says, and one thing to check before believing all of it.**

PFC is dominated by `place` and `speed`, with `time_from_reward` and `acceleration` behind
them. The task-abstract variables — `task_state`, `time_since_A`, `time_to_A`,
`progress_since_A` — sit at 10–11% significant against a 5% chance line, with negative Δr².

**The thing to check:** `goal_progress` has a median Δr² of ~0.00000 yet 47% of neurons are
called significant, and `distance_from_reward` is *negative* (−0.0004) with 42% significant.
A significant fraction that far above chance alongside a null or negative median effect
means either a widespread but vanishingly small effect, or a permutation null that is too
narrow for those particular regressors. The within-session circular shift preserves
autocorrelation but may not fully destroy alignment for slowly-varying variables — the same
issue that made the old in-sample place test saturate at 80%. Worth checking the null width
per regressor (`null_mean`, `null_p95` are stored in the results) before quoting those two
numbers.

### Both effect sizes: $\Delta R^2$ and CPD

Every plotting function takes `value='delta_r2_cv'` (default) or `value='cpd_cv'`. They share
a numerator — held-out $\Delta$RSS — and differ only in denominator:

$$\Delta R^2_g = \frac{\Delta \mathrm{RSS}_g}{\mathrm{TSS}} \qquad\qquad
\mathrm{CPD}_g = \frac{\Delta \mathrm{RSS}_g}{\mathrm{RSS}_{\text{reduced},g}}$$

So $\mathrm{CPD} \ge \Delta R^2$ always, and the gap grows with how well the reduced model
still fits.

**On this data the choice barely matters, and I over-warned about it earlier.** The 12×
inflation I demonstrated in simulation needs a model that explains a lot; here $R^2_{cv}$ is
0.03–0.09, so $\mathrm{RSS}_{\text{reduced}} \approx \mathrm{TSS}$ and the two agree to within
about 8% (place: 0.01283 vs 0.01380 in LEC). Use $\Delta R^2$ for comparing regressors — the
common denominator is still the principled choice, and it becomes the *only* defensible one if
the fit ever improves — but no conclusion here turns on it.

**What does not change with `value`: the significance test.** `p_cv` is computed on CPD
regardless, so `frac_sig` is identical in both tables. See the significance section above for
why that is a mismatch worth fixing.

In [ ]:
for val in P.VALUE_OPTIONS:          # raw AND bias-corrected
    short = P._VALUE_SHORT[val]
    P.plot_regressor_ranking(cv_pfc, value=val,
                             out_path=f'../data/figures/pfc_glm/pfc_ranking_{short}.pdf')
    P.plot_neuron_heatmap(cv_pfc, value=val,
                          out_path=f'../data/figures/pfc_glm/pfc_heatmap_{short}.pdf')
    print(f'--- {short} ---')
    display(P.summary_table(cv_pfc, value=val).round(5))

The two side by side, per neuron per regressor. The dashed line is $y=x$;
distance above it is the inflation. Colour is regressor family.

In [ ]:
P.plot_cpd_vs_delta_r2(cv_pfc, out_path=f'../data/figures/pfc_glm/pfc_cpd_vs_delta_r2.pdf');

### Bias-corrected effect sizes

Held-out $\Delta R^2$ and CPD both carry a **downward penalty proportional to a regressor's
column count**. The full model has *k* more parameters than the reduced one; if the regressor
explains nothing those *k* parameters fit training noise, and noise-fitted parameters *add*
held-out error. So a regressor explaining nothing scores about $-k\sigma^2/\text{denominator}$
— **not zero**.

Measured here: `corr(n_cols, null centre) = -0.838`, and all 16 nulls sit below zero.

**Zero is not the reference; the null centre is.** The permutation null measures that penalty
empirically per neuron (under permutation the regressor explains nothing by construction), so
subtracting it gives an unbiased effect:

| | under H₀ | with signal *s* |
|---|---|---|
| observed | $-k\sigma^2/\text{den}$ | $(s-k\sigma^2)/\text{den}$ |
| null mean | $-k\sigma^2/\text{den}$ | $-k\sigma^2/\text{den}$ |
| **corrected** | **≈ 0** | **≈ $s/\text{den}$** |

Correcting moves **14 of 16 regressors** in the ranking. `head_direction` (35 columns, the
largest penalty in the design) rises from 4th to 2nd; `poke_rewarded` (1 column, almost no
penalty) falls from 5th to 10th.

Two limits worth holding onto. It removes the *parameter penalty*, not the **capacity
advantage** — `corr(n_cols, corrected) = +0.517` remains, and a 35-column block genuinely can
capture more real structure than a 1-column indicator. And the correction inherits whichever
null it is centred on: these fits carry only the legacy shuffle-null key, so `resolve_value`
falls back to that. All three nulls measured the same centre on synthetics, but the
Freedman-Lane refit is what the plan specifies.

In [ ]:
import glm_analysis_v2 as _g, w1_refit as _w
_groups, _ = _g._resolve_regressor_groups(_w.SECTIONS['all_regressors']['regressors'],
                                          gp_n_bins=10, parameterization='reference_coded')
n_cols = {k: len(v) for k, v in _groups.items()}

P.plot_bias_correction(cv_pfc, n_cols=n_cols,
                       out_path=f'../data/figures/pfc_glm/pfc_bias_correction.pdf');
display(P.summary_table(cv_pfc, value='delta_r2_corrected').round(5))